# Baseline training notebook (ResNet50 on log-mel)
This notebook is a compact, runnable baseline intended for fast iteration. It trains a small ResNet50-based classifier on cached log-mel patches.
Keep epochs small for quick runs; use the config cell to scale up.

In [ ]:
# Imports and reproducibility
import os, sys, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa, soundfile as sf
from torchvision import models, transforms

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed);
    torch.cuda.manual_seed_all(seed)
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Config (edit for larger runs)
ROOT = os.path.abspath(os.path.join('..','..'))  # repo root relative to this notebook path
DATA_ROOT = os.path.join(ROOT, 'data','raw')
TRAIN_CSV = os.path.join(DATA_ROOT, 'train.csv')
TAXONOMY_CSV = os.path.join(DATA_ROOT, 'taxonomy.csv')
MEL_CACHE = os.path.join(ROOT, 'cached_mels')
os.makedirs(MEL_CACHE, exist_ok=True)
CFG = dict(
    sr=32000, n_mels=128, duration=5.0, hop_length=320, n_fft=2048,
    batch_size=32, epochs=2, lr=1e-4, num_workers=4,
)
CFG['samples'] = int(CFG['sr'] * CFG['duration'])
CFG

In [ ]:
# Quick dataset peek
df = pd.read_csv(TRAIN_CSV)
tax = pd.read_csv(TAXONOMY_CSV)
print('train rows', len(df))
display(df.head())
display(tax.head())

In [ ]:
# Mel extraction + caching helper
import hashlib, pathlib
def load_audio(path, sr=CFG['sr'], samples=CFG['samples']):
    y, _ = librosa.load(path, sr=sr, mono=True, res_type='kaiser_fast')
    if len(y) < samples:
        y = np.pad(y, (0, max(0, samples - len(y))))
    else:
        y = y[:samples]
    return y
def mel_spectrogram(y, sr=CFG['sr']):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=CFG['n_fft'], hop_length=CFG['hop_length'], n_mels=CFG['n_mels'])
    Sdb = librosa.power_to_db(S, ref=np.max)
    return Sdb.astype(np.float32)
def cached_mel_path(filepath):
    h = hashlib.sha1(filepath.encode()).hexdigest()
    return os.path.join(MEL_CACHE, h + '.npy')
def get_mel_for_file(filepath):
    cache = cached_mel_path(filepath)
    if os.path.exists(cache):
        return np.load(cache)
    y = load_audio(filepath)
    m = mel_spectrogram(y)
    np.save(cache, m)
    return m

In [ ]:
# Simple PyTorch dataset for single-label files (expects train.csv filename column contains relative path)
class MelDataset(Dataset):
    def __init__(self, df, root, transform=None):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.transform = transform
        self.labels = self.df['primary_label'].astype(str).values
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = os.path.join(self.root, row['filename'])
        mel = get_mel_for_file(filepath)  # (n_mels, time)
        # convert to 1xHxW tensor
        x = torch.tensor(mel)[None,:,:]
        if self.transform: x = self.transform(x)
        y = int(row['primary_label'])
        return x, y
# small transform: normalization
transform = None
ds = MelDataset(df.sample(500, random_state=42).head(200), os.path.join(DATA_ROOT,'train_audio'))
loader = DataLoader(ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=0)
next(iter(loader))[0].shape

In [ ]:
# Model: adapt torchvision ResNet50 for 1-channel input and num_classes = len(taxonomy) (quick)
num_classes = tax.shape[0]
model = models.resnet50(pretrained=True)
# adapt first conv (3->1) by averaging weights
w = model.conv1.weight.data
model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.conv1.weight.data = w.mean(dim=1, keepdim=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
print('model ready, classes=', num_classes)

In [ ]:
# Tiny training loop (for quick iteration)
opt = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-5)
crit = nn.CrossEntropyLoss()
for epoch in range(CFG['epochs']):
    model.train()
    running = 0.0
    for i, (x,y) in enumerate(loader):
        x = x.to(device).float()
        y = y.to(device)
        # resize mel time -> make square-ish for ResNet by simple interpolation
        x = torch.nn.functional.interpolate(x, size=(CFG['n_mels'], CFG['n_mels']))
        opt.zero_grad()
        logits = model(x)
        loss = crit(logits, y)
        loss.backward(); opt.step()
        running += loss.item()
        if i % 10 == 0:
            print(f'Epoch {epoch} iter {i} loss {running/(i+1):.4f}')
    print('epoch', epoch, 'done')
# Save a quick checkpoint
ckpt = os.path.join(ROOT, 'models','baseline_resnet50.pt')
os.makedirs(os.path.dirname(ckpt), exist_ok=True)
torch.save({'model_state': model.state_dict()}, ckpt)
print('saved', ckpt)